In [2]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)


d:\GenAi\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [4]:
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

In [9]:
store={}
def get_session_history(session_id:str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful assistant. Remember everything the user tells you throughout the conversation."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def chat(user_input: str, session_id: str = "default") -> str:
    response = chain_with_history.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}},
    )
    return response.content

def main():
    print("Chatbot with Memory (type 'quit' to exit, 'clear' to reset history)")
    print("-" * 60)
    session_id = "user_session_1"

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            continue
        if user_input.lower() == "quit":
            print("Goodbye!")
            break
        if user_input.lower() == "clear":
            store.pop(session_id, None)
            print("Chat history cleared.")
            continue

        response = chat(user_input, session_id)
        print(f"Bot: {response}\n")


In [10]:

if __name__ == "__main__":
    main()

Chatbot with Memory (type 'quit' to exit, 'clear' to reset history)
------------------------------------------------------------
Bot: Hey there! Not much—just here and ready to help out. What's on your mind today?

Chat history cleared.
Bot: Alright, goodbye! If you need anything else later, feel free to reach out. Take care!



KeyboardInterrupt: Interrupted by user